# Day 36 — Feature selection & PCA
Objectives:
- PCA intuition and variance explained.
- Visualize components.
- Simple univariate selection with SelectKBest.

In [ ]:
from sklearn.decomposition import PCA
from sklearn.datasets import load_iris
import matplotlib.pyplot as plt
X,y = load_iris(return_X_y=True)
pca = PCA(n_components=2).fit(X)
X2 = pca.transform(X)
plt.scatter(X2[:,0], X2[:,1], c=y); plt.title('PCA(2)'); plt.show()
pca.explained_variance_ratio_


## How to use this notebook

Select the `Python (ds60sqlpy)` kernel, start at the first cell, and
write each prediction before execution. Keep attempts in the
provided scratch cell or new cells. Restart the kernel and run from
the top before calling the work reproducible.

## Concept lab — PCA geometry, scaling, variance, and reconstruction

### Mental model

Principal component analysis (PCA) rotates centered feature space to
new orthogonal axes ordered by captured variance. A component is a
direction; a score is an observation projected onto that direction;
a loading describes how original features contribute. PCA does not
know the target and does not discover causal factors.

Variance is measured in squared feature units. If one feature is
measured on a much larger numeric scale, unscaled PCA may devote its
first component to that unit rather than to the structure you care
about. Component count is therefore a modeling choice connected to
scaling, reconstruction, and downstream performance.

### Read the API before running it

- **`StandardScaler()`:** centers and scales features when equalized variance, rather than raw units, is the intended geometry.
- **`PCA(n_components=...).fit(X_train)`:** learns training-only means, axes, and explained variance.
- **`.transform()` / `.inverse_transform()`:** moves between feature and component spaces, enabling reconstruction-error checks.

For every call, identify input data, learned state, returned value,
and a check that can fail. That habit prevents a successful cell
from being mistaken for a correct analysis.

### Focused example A — observe how units change the first component

**Predict first:** write down the expected shape, type, ordering, or
direction of the result. Then run the next cell.

**Assumption:** Equal standard-deviation weighting is scientifically appropriate; scaling is not automatically correct.

In [ ]:
import numpy as np
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

rng = np.random.default_rng(3601)
signal = rng.normal(size=300)
X = np.column_stack([
    signal + rng.normal(scale=0.1, size=300),
    1_000 * rng.normal(size=300),
])

raw = PCA(n_components=1).fit(X)
scaled = PCA(n_components=1).fit(StandardScaler().fit_transform(X))
print({"raw_loading": raw.components_[0],
       "scaled_loading": scaled.components_[0]})

**Expected observation:** Raw PCA points almost entirely along the large-unit feature; scaling changes the geometry and loadings.

Do not force exact equality for estimates based on samples. Record
the seed, sample size, tolerance, and metric where they matter.

### Focused example B — connect retained components to reconstruction error

This example changes one important condition. Predict how and why
the result should differ from Example A.

**Assumption:** Squared reconstruction error is a useful measure of the information relevant to this task.

In [ ]:
import numpy as np
from sklearn.datasets import load_iris
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

X = StandardScaler().fit_transform(load_iris().data)
errors = {}
for components in (1, 2, 3, 4):
    pca = PCA(n_components=components).fit(X)
    reconstructed = pca.inverse_transform(pca.transform(X))
    errors[components] = np.mean((X - reconstructed) ** 2)
print(errors)
assert all(errors[k] >= errors[k + 1] for k in (1, 2, 3))

**Expected observation:** Reconstruction error cannot increase as more components are retained and reaches numerical zero with all components.

### Debugging and practice ramp

**Common mistake:** Choosing components from the full dataset and then cross-validating a downstream model.

**Diagnostic:** Put scaling and PCA inside the evaluated pipeline; inspect component shapes, cumulative variance, and reconstruction on training versus validation.

| Stage | Action | Evidence |
|---|---|---|
| Recall | Define PCA geometry, scaling, variance, and reconstruction in your own words and identify its input and output. | A definition that does not rely on the library name. |
| Predict | Predict the examples before execution, including shape and direction. | A written prediction and an explanation of any mismatch. |
| Implement | Recreate one example with a changed but valid input. | Code plus an assertion for the central invariant. |
| Debug | Trigger the named mistake or edge case intentionally. | The observed symptom and the smallest diagnostic that isolates it. |
| Transfer | Apply the idea to a different local dataset or decision. | A stated assumption, metric, and reason the method is suitable. |

**Stop condition:** Do not label a component semantically from one large loading without checking units, correlations, sign ambiguity, and stability.

Continue to the numbered practice only after you can explain both
examples without rereading their code.

## Learner exercises and progressive hints

1. Plot cumulative explained variance and choose a component count.

**Verify:** Practice 1 — PCA geometry, scaling, variance, and reconstruction — fit PCA on training data only, print cumulative explained variance by component, and choose the smallest count reaching a declared threshold such as 95%; assert transformed train/test column counts match that choice.

2. Compare PCA before and after feature standardization.

**Verify:** Practice 2 — PCA geometry, scaling, variance, and reconstruction — on one frozen split, report feature scales and PCA explained-variance ratios before and after StandardScaler; verify each scaler/PCA pair is fitted on training rows only.

3. Try `SelectKBest` on a classification dataset and compare its validated
   performance with PCA.

**Verify:** Practice 3 — PCA geometry, scaling, variance, and reconstruction — evaluate SelectKBest and scaled PCA with identical folds, component/feature counts, estimator, and metric; print every fold score plus mean/std and keep the final test labels unopened.

### Progressive hints

1. Fit all possible components first, then use `np.cumsum`. Declare a threshold
   such as 95% before looking for the first component count that crosses it.
2. Compare both the variance ratios and two-dimensional scatter plots. Iris
   features use different physical units and spreads.
3. Put each alternative in its own pipeline. `f_classif` sees the target; PCA
   does not, so the comparison includes an interpretability tradeoff.

### Additional mastery practice

Use dimensionality reduction as a fitted transformation with measurable information loss. Keep scaling and supervised feature selection inside validation.

Predict or plan before you run code. Use the hint only after an honest
attempt, and record the evidence that would prove your result correct.

4. **Reconstruction analysis:** Fit scaled PCA with several component counts, inverse-transform the representations, and plot mean squared reconstruction error versus retained components.
   **Progressive hint:** Call transform then inverse_transform on the same fitted PCA and compare in scaled space. Error should not increase as components are added.

**Verify:** Reconstruction analysis — for each retained-component count, print reconstruction MSE and save the labeled curve; assert inverse-transformed shape equals the scaled input shape and MSE is non-increasing up to floating-point tolerance.

5. **Interpretation edge case:** Fit PCA twice to equivalent data and explain why a component and all of its loadings may appear with the opposite sign while the projection remains equivalent.
   **Progressive hint:** Eigenvectors are direction axes: v and -v describe the same axis. Compare subspaces or absolute loading patterns, not raw signs alone.

**Verify:** Interpretation edge case — fit the equivalent PCA inputs, align component signs by dot product, and assert transformed coordinates/loadings match after sign alignment within 1e-10 while explained-variance ratios are unchanged.

6. **Leakage debugging:** Create a dataset with many noise features, run SelectKBest once before cross-validation, and then correctly inside a Pipeline. Explain the expected score difference.
   **Progressive hint:** Selection performed globally can choose noise features that happen to correlate with all labels, including validation labels.

**Verify:** Leakage debugging — print fold scores for SelectKBest fitted globally and inside Pipeline on the same seeded noise dataset; assert the pipeline owns fit within each fold and retain the observed optimism gap without promising its exact size.

Before opening the reference solution, explain the relevant assumption,
failure mode, and validation check for every answer.

In [ ]:
# Expanded mastery lab scratch space
#
# Keep the official solution closed until you have attempted each task.
# Add small assertions, shape checks, or metric comparisons as evidence.

# Practice 4 — Reconstruction analysis


# Practice 5 — Interpretation edge case


# Practice 6 — Leakage debugging
